In [ ]:
##环境准备
import os, shutil
os.environ["FOLDX_BINARY"] = "/home/hkl/foldx5/foldx"
from pyfoldx.structure import Structure
import pandas as pd

os.makedirs("scan_output", exist_ok=True)
if not os.path.exists("molecules"):
    shutil.copytree("/home/hkl/foldx5/molecules", "molecules")
print("环境就绪")

In [ ]:
##加载并修复结构
st = Structure("complex", "yeats2_BTN_fixed.pdb")
st.repair()
print("✅ Repair 完成")

In [ ]:
##看清有哪些链（关键，别跳）
for chain in st.data:
    print("链", chain, ":", [(r, st.data[chain][r].code) for r in st.data[chain]])

In [ ]:
##野生型结合能
ie = st.getInterfaceEnergy()
display(ie)

In [ ]:
##复合物内饱和扫描
# ===== 扫描 YEATS2 口袋残基（界面残基，≤5 Å）=====
TARGET_CHAIN = "A"   # YEATS2

def atoms_of(chain_id):
    pts = []
    for l in open("yeats2_BTN_fixed.pdb"):
        if l.startswith(("ATOM","HETATM")) and l[21] == chain_id:
            pts.append((int(l[22:26]), float(l[30:38]), float(l[38:46]), float(l[46:54])))
    return pts

A_atoms  = atoms_of("A")
CD_atoms = atoms_of("C") + atoms_of("D")   # C=肽, D=BTN

interface = set()
for ra, xa, ya, za in A_atoms:
    for rc, xc, yc, zc in CD_atoms:
        if (xa-xc)**2 + (ya-yc)**2 + (za-zc)**2 <= 25.0:  # 5 Å²
            interface.add(ra)
            break

targets = sorted(interface)
print(f"YEATS2 口袋/界面残基 {len(targets)} 个:", targets)

scan_complex = {}
for pos in targets:
    try:
        df = st.positionScan([pos], TARGET_CHAIN)
        scan_complex[pos] = df
        df.to_csv(f"scan_output/complex_A{pos}.csv")
        print(f"✅ A{pos}")
    except Exception as e:
        print(f"⚠️ A{pos}: {e}")

In [ ]:
##提取游离 YEATS2
lines = [l.rstrip("\n") for l in open("yeats2_BTN_fixed.pdb")
         if l.startswith(("ATOM","HETATM")) and l[21] == "A"]
with open("scan_output/protein_alone.pdb","w") as f:
    f.write("\n".join(lines) + "\nEND\n")
prot = Structure("protein", "scan_output/protein_alone.pdb")
print("✅ 游离 YEATS2 已提取:", len(lines), "行")

In [ ]:
##游离态扫描
scan_prot = {}
for pos in targets:
    try:
        df = prot.positionScan([pos], TARGET_CHAIN)
        scan_prot[pos] = df
        df.to_csv(f"scan_output/prot_A{pos}.csv")
        print(f"✅ 游离 A{pos}")
    except Exception as e:
        print(f"⚠️ 游离 A{pos}: {e}")

In [ ]:
##看一个结果的结构（决定怎么排序）
example_pos = targets[0]
print("复合物扫描结果示例:")
display(scan_complex[example_pos])
print("游离YEATS2扫描结果示例:")
display(scan_prot[example_pos])      # ← scan_pep 改成 scan_prot

In [ ]:
# ===== 计算 ΔΔG_bind 并排序（YEATS2 口袋版）=====
AA1 = list("ARNDCQEGHILKMFPSTWYV")
three2one = {"ALA":"A","ARG":"R","ASN":"N","ASP":"D","CYS":"C","GLN":"Q","GLU":"E",
             "GLY":"G","HIS":"H","ILE":"I","LEU":"L","LYS":"K","MET":"M","PHE":"F",
             "PRO":"P","SER":"S","THR":"T","TRP":"W","TYR":"Y","VAL":"V"}
rows = []
for pos in targets:
    c = scan_complex[pos].iloc[:, 0]   # 复合物中的 ΔΔG
    p = scan_prot[pos].iloc[:, 0]    # 游离 YEATS2 中的 ΔΔG
    wt = three2one[st.data["A"][pos].code]
    for aa in AA1:
        if aa in c.index and aa in p.index and aa != wt:
            rows.append({
                "mutation": f"{wt}A{pos}{aa}",
                "ddG_complex": round(c[aa], 3),
                "ddG_protein": round(p[aa], 3),
                "ddG_bind": round(c[aa]-p[aa], 3),   # 负值 = 亲和力提升
            })
rank = pd.DataFrame(rows).sort_values("ddG_bind").reset_index(drop=True)
print("==== 亲和力提升最大的前 20 个突变（ΔΔG_bind 越负越好）====")
display(rank.head(20))
rank.to_csv("scan_output/ddG_bind_ranking_protein.csv", index=False)
print("✅ 完整排序已存 scan_output/ddG_bind_ranking_protein.csv，共", len(rank), "个突变")

In [ ]:
final = rank[(rank.ddG_bind < -0.5) &
             (rank.ddG_protein < 0.5) &
             (rank.ddG_complex < 0.5)].sort_values("ddG_bind")
display(final)

BuildModel 复算

In [ ]:
# ===== BuildModel 复算（绕过 pyfoldx 的 mutate bug，直接调 FoldX）=====
import subprocess, os, shutil, glob

def totalE(st):
    df = st.getTotalEnergy()
    for col in ["Total", "total", "TOTAL"]:
        if col in getattr(df, "columns", []):
            return float(df[col].iloc[0])
    return float(df.iloc[0, -1])

def run_buildmodel(mutation, pdb_in="yeats2_BTN_fixed.pdb", tag=None, nruns=1):
    """自己写带分号的突变列表，直接调 FoldX BuildModel"""
    tag = tag or mutation.replace(",", "_")
    wd = f"bm_{tag}"
    os.makedirs(wd, exist_ok=True)
    shutil.copy(pdb_in, f"{wd}/tmp.pdb")
    if os.path.isdir("molecules"):
        shutil.copytree("molecules", f"{wd}/molecules", dirs_exist_ok=True)
    with open(f"{wd}/individual_list.txt", "w") as f:
        f.write(mutation + ";\n")
    r = subprocess.run(
        [os.path.expanduser("~/foldx5/foldx"), "--command=BuildModel",
         "--pdb=tmp.pdb", "--mutant-file=individual_list.txt",
         f"--numberOfRuns={nruns}", "--output-dir=."],
        cwd=wd, capture_output=True, text=True)
    if r.returncode != 0 or "successfully finished" not in r.stdout:
        print(r.stdout[-1500:]); print(r.stderr[-1500:])
        raise RuntimeError(f"{mutation} BuildModel 失败")
    wt = glob.glob(f"{wd}/WT_*.pdb")
    mut = [p for p in glob.glob(f"{wd}/*.pdb")
           if os.path.basename(p) != "tmp.pdb" and "WT_" not in os.path.basename(p)]
    assert wt and mut, "输出PDB不全"
    print(f"✅ {mutation}: WT模型={wt[0]} 突变模型={mut[0]}")
    return wt[0], mut[0]

CANDIDATES = ["HA259D", "WA282P", "SA261E", "EA284P"]
rows = []
for mut in CANDIDATES:
    print(f"==== {mut} ====")
    wt_c, mut_c = run_buildmodel(mut, "yeats2_BTN_fixed.pdb", tag=f"{mut}_cx")
    d_cx = totalE(Structure("m", mut_c)) - totalE(Structure("w", wt_c))
    wt_p, mut_p = run_buildmodel(mut, "scan_output/protein_alone.pdb", tag=f"{mut}_pr")
    d_pr = totalE(Structure("m2", mut_p)) - totalE(Structure("w2", wt_p))
    d_bind = d_cx - d_pr
    print(f"  复合物ΔΔG={d_cx:.2f}  蛋白ΔΔG={d_pr:.2f}  ΔΔG_bind={d_bind:.2f}\n")
    rows.append({"mutation": mut, "ddG_complex": round(d_cx,3),
                 "ddG_protein": round(d_pr,3), "ddG_bind_BuildModel": round(d_bind,3)})

verify = pd.DataFrame(rows)
compare = verify.merge(rank[["mutation","ddG_bind"]], on="mutation", how="left")
compare.columns = ["mutation","ddG_complex","ddG_protein","ddG_bind_BuildModel","ddG_bind_Scan"]
display(compare)
compare.to_csv("scan_output/buildmodel_verification.csv", index=False)
print("✅ 已存 scan_output/buildmodel_verification.csv")

给复合物模型（_cx）注 BTN，游离蛋白（_pr）不注

In [ ]:
# ===== BuildModel 复算 v3：直接 FoldX Stability（绕过 pyfoldx 写文件丢 BTN 的 bug）=====
def foldx_stability(pdb, wd):
    """直接调用 FoldX Stability 计算总能量（保留 HETATM/BTN）。
    输入: pdb(str) 结构文件; wd(str) 工作目录  输出: float 总能量"""
    os.makedirs(wd, exist_ok=True)
    shutil.copy(pdb, f"{wd}/tmp.pdb")
    if os.path.isdir("molecules"):
        shutil.copytree("molecules", f"{wd}/molecules", dirs_exist_ok=True)
    r = subprocess.run([os.path.expanduser("~/foldx5/foldx"), "--command=Stability",
                        "--pdb=tmp.pdb", "--output-dir=."],
                       cwd=wd, capture_output=True, text=True)
    m = re.search(r"Total\s*[:=]?\s*(-?\d+\.\d+)", r.stdout)
    if not m:
        raise RuntimeError(f"Stability 解析失败:\n{r.stdout[-1200:]}")
    return float(m.group(1))

rows = []
for mut in ["HA259D", "WA282P", "SA261E", "EA284P"]:
    wt_c  = inject_btn(f"bm_{mut}_cx/WT_tmp_1.pdb")
    mut_c = inject_btn(f"bm_{mut}_cx/tmp_1.pdb")
    e_wt_c  = foldx_stability(wt_c,  f"stab_{mut}_wtcx")
    e_mut_c = foldx_stability(mut_c, f"stab_{mut}_mtcx")
    e_wt_p  = foldx_stability(f"bm_{mut}_pr/WT_tmp_1.pdb",  f"stab_{mut}_wtpr")
    e_mut_p = foldx_stability(f"bm_{mut}_pr/tmp_1.pdb",     f"stab_{mut}_mtpr")
    d_cx, d_pr = e_mut_c - e_wt_c, e_mut_p - e_wt_p
    d_bind = d_cx - d_pr
    print(f"{mut}: 复合物(真含BTN)={d_cx:.2f}  蛋白={d_pr:.2f}  ΔΔG_bind={d_bind:.2f}")
    rows.append({"mutation": mut, "ddG_complex": round(d_cx,3),
                 "ddG_protein": round(d_pr,3), "ddG_bind_withBTN": round(d_bind,3)})

fx3 = pd.DataFrame(rows).merge(rank[["mutation","ddG_bind"]], on="mutation") \
        .rename(columns={"ddG_bind": "ddG_bind_Scan"})
display(fx3)
fx3.to_csv("scan_output/ddG_withBTN.csv", index=False)
print("✅ 已用 Stability-直接调用版 覆盖 ddG_withBTN.csv")

In [ ]:
import math
def all_res_atoms(pdb):
    out = {}
    for l in open(pdb):
        if l.startswith(("ATOM","HETATM")):
            key = (l[21], int(l[22:26]))
            out.setdefault(key, {})[l[12:16].strip()] = (
                float(l[30:38]), float(l[38:46]), float(l[46:54]))
    return out

def mind(x,y): return min(math.dist(p,q) for p in x.values() for q in y.values())

wt  = "bm_SA261E_cx/WT_tmp_1.pdb"
mut = "bm_SA261E_cx/tmp_1.pdb"
wt_res, mut_res = all_res_atoms(wt), all_res_atoms(mut)

og = wt_res[("A",261)]["OG"]
partners = set()
print("WT 中 SER261-OG 的邻居（<3.5 Å）：")
for (chain,resid), atoms in wt_res.items():
    if (chain,resid) == ("A",261): continue
    for aname,p in atoms.items():
        if math.dist(og,p) < 3.5:
            partners.add((chain,resid))
            print(f"  {aname:4s} 链{chain} {resid}号  d={math.dist(og,p):.2f} Å")

In [ ]:
# ===== SA261E：验证"断氢键 → 形成新疏水作用力" =====
BB = {"N","CA","C","O"}   # 主链原子，排除

def sc_carbons(res_atoms):
    """残基的侧链碳原子（疏水作用的主要载体）"""
    return {k:v for k,v in res_atoms.items() if k not in BB and k[0]=="C"}

def min_pair_dist(atoms1, atoms2):
    return min(math.dist(p,q) for p in atoms1.values() for q in atoms2.values())

wt  = "bm_SA261E_cx/WT_tmp_1.pdb"
mut = "bm_SA261E_cx/tmp_1.pdb"
wt_res, mut_res = all_res_atoms(wt), all_res_atoms(mut)

# E261(突变) / S261(WT) 的侧链碳
wt_sc  = sc_carbons(wt_res[("A",261)])
mut_sc = sc_carbons(mut_res[("A",261)])
print(f"261侧链碳: WT={sorted(wt_sc)} → 突变={sorted(mut_sc)}\n")

# 扫所有其他残基，找"WT>5.0 且 突变<4.5"的新疏水接触
print("新形成的疏水接触（WT没有或更远的）：")
print(f"{'伙伴':>10} {'WT距离':>8} {'突变距离':>8}")
found = False
for (chain,resid), atoms in mut_res.items():
    if (chain,resid) == ("A",261): continue
    mut_c = sc_carbons(atoms)
    if not mut_c: continue
    d_mut = min_pair_dist(mut_sc, mut_c)
    if d_mut > 4.5: continue
    d_wt  = min_pair_dist(wt_sc, sc_carbons(wt_res[(chain,resid)])) if (chain,resid) in wt_res else 99.0
    if d_wt - d_mut > 0.8:      # 显著拉近才算"新形成"
        found = True
        print(f"  链{chain} {resid:<5} {d_wt:6.2f}Å {d_mut:8.2f}Å   (拉近{d_wt-d_mut:+.2f}Å)")

if not found:
    print("  （没有检测到明显的新疏水接触）")

# 顺带看断裂的氢键伙伴现在离多远（用上一段代码的 partners 变量）
print("\n原氢键伙伴现状：")
for (chain,resid) in sorted(partners):
    print(f"  261-链{chain}{resid}: WT {mind(wt_res[('A',261)], wt_res[(chain,resid)]):.2f} Å "
          f"→ 突变 {mind(mut_res[('A',261)], mut_res[(chain,resid)]):.2f} Å")

In [ ]:
# 专门测 E261 羧基氧 ↔ K12-NZ（潜在盐桥）
for tag, r in [("WT", wt_res), ("突变", mut_res)]:
    r261, k12 = r[("A",261)], r[("C",12)]
    if "OE1" in r261:
        d = min(math.dist(r261["OE1"], k12["NZ"]), math.dist(r261["OE2"], k12["NZ"]))
        print(f"{tag}: 261-OE···K12-NZ = {d:.2f} Å   {'← 成盐桥!' if d < 4.0 else ''}")
    else:
        print(f"{tag}: 261-OG···K12-NZ = {math.dist(r261['OG'], k12['NZ']):.2f} Å")

In [ ]:
# 看两个模型里实际存在哪些链
for tag, r in [("WT", wt_res), ("突变", mut_res)]:
    chains = {}
    for (c,resid) in r:
        chains.setdefault(c, []).append(resid)
    print(tag, {c: f"{min(v)}-{max(v)} ({len(v)}个残基)" for c,v in sorted(chains.items())})

In [ ]:
# BTN 坐标取自输入文件（bm_SA261E_cx/tmp.pdb 是 yeats2_BTN_fixed 的拷贝，含BTN）
btn_full = all_res_atoms("bm_SA261E_cx/tmp.pdb")
btno = {k: v for k, v in btn_full[("D",1)].items() if k in neg}

for tag, r261 in [("WT(OG)",   {k:v for k,v in wt_res[("A",261)].items()  if k=="OG"}),
                  ("突变(OE)", {k:v for k,v in mut_res[("A",261)].items() if k in ("OE1","OE2")})]:
    print(tag)
    for bn, bp in btno.items():
        d = min(math.dist(o, bp) for o in r261.values())
        print(f"  261侧链氧···BTN {bn}: {d:.2f} Å {'⚠️ 偏近' if d < 4.0 else ''}")

In [ ]:
# 以 EA284P 为例：突变体 vs WT 模型中 284 与多肽/D310 的距离对比
import math
def resid_atoms(pdb, chain, resid):
    out = {}
    for l in open(pdb):
        if l.startswith(("ATOM","HETATM")) and l[21]==chain and int(l[22:26])==resid:
            out[l[12:16].strip()] = (float(l[30:38]),float(l[38:46]),float(l[46:54]))
    return out
def mind(x,y): return min(math.dist(p,q) for p in x.values() for q in y.values())

for tag, pdb in [("WT模型","bm_EA284P_cx/WT_tmp_1.pdb"), ("突变模型","bm_EA284P_cx/tmp_1.pdb")]:
    r284 = resid_atoms(pdb,"A",284)
    print(tag, "284-肽:", round(mind(r284, resid_atoms(pdb,"C",12)),2), "Å",
               "| 284-D310:", round(mind(r284, resid_atoms(pdb,"A",310)),2), "Å")